<a href="https://colab.research.google.com/github/aryank2074-ai/llamaindex-rag-chatbot/blob/main/Llama_index_and_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# LlamaIndex Complete RAG Chatbot
# Pipeline: Documents -> Chunking -> Embeddings -> Vector Index
#           -> Retrieval -> LLM -> Answer
#
# This notebook builds a small Retrieval-Augmented Generation (RAG)
# system end-to-end: it loads local text documents, splits them into
# chunks, embeds those chunks, stores them in a vector index, and
# then answers natural-language questions by retrieving the most
# relevant chunks and passing them to an LLM as context.
# ================================================================
print("🚀 LlamaIndex RAG Chatbot Tutorial")
print("=" * 60)


In [ ]:
# Install the LlamaIndex core library plus the integrations we need:
# - llama-index-llms-openai        -> lets LlamaIndex call OpenAI chat models
# - llama-index-embeddings-openai  -> lets LlamaIndex call OpenAI embedding models
# - llama-index-vector-stores-chroma / chromadb -> optional persistent vector store
# - pypdf                          -> lets SimpleDirectoryReader read PDF files too
# - python-dotenv                  -> lets us load API keys from a .env file
%pip install -q llama-index
%pip install -q llama-index-llms-openai
%pip install -q llama-index-embeddings-openai
%pip install -q llama-index-vector-stores-chroma
%pip install -q chromadb
%pip install -q pypdf
%pip install -q python-dotenv


In [ ]:
import importlib.metadata

from dotenv import load_dotenv

# Core LlamaIndex building blocks:
# - VectorStoreIndex     : turns document chunks into a searchable vector index
# - SimpleDirectoryReader: loads all files from a folder into Document objects
# - Settings              : global config object (which LLM/embedding model to use)
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
)

# LLM and embedding model wrappers for OpenAI
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

# Load any environment variables defined in a local .env file (e.g. OPENAI_API_KEY)
load_dotenv()

print(
    "LlamaIndex version:",
    importlib.metadata.version("llama-index")
)

print("Libraries imported successfully!")


In [ ]:
# Quick sanity check: print the installed version of every package this
# notebook depends on. Useful for debugging "it works on my machine" issues,
# since LlamaIndex's API has changed across versions.
import importlib.metadata

packages = [
    "llama-index",
    "llama-index-core",
    "llama-index-llms-openai",
    "llama-index-embeddings-openai",
    "llama-index-vector-stores-chroma",
    "chromadb",
]

for package in packages:
    try:
        print(f"{package}: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")


In [ ]:
import os
from getpass import getpass

# Only prompt for the API key if it isn't already set (e.g. via .env or
# a previous cell run). getpass() hides the input so the key never gets
# printed to the notebook output or saved in cell history.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API Key: "
    )

print("API key configured successfully.")


In [ ]:
# LlamaIndex
#      │
#      └── LLM
#           │
#           └── OpenAI
#
# The LLM is the "brain" that reads retrieved context and generates the
# final natural-language answer. temperature=0 makes it deterministic
# (same question -> same answer), which is ideal for factual Q&A over
# our own documents rather than creative writing.
llm = OpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# Settings is a global config object: once set here, every LlamaIndex
# component (indexing, querying, etc.) automatically uses this LLM
# without us having to pass it around manually.
Settings.llm = llm

print("LLM configured successfully.")
print("Model:", llm.model)
print("Temperature:", 0)  # Configured model to deliver a relatively deterministic response


In [ ]:
# Configure Embedding Model
#
# Text
#  ↓
# Embedding Model
#  ↓
# Vector
#
# "Machine learning is a subset of AI"
#         ↓
# [0.021, -0.182, 0.731, ...]
#
# The embedding model converts text into a fixed-length numeric vector
# that captures its meaning. Chunks with similar meaning end up with
# similar vectors, which is what makes semantic search possible later.
embed_model = OpenAIEmbedding(
    model="text-embedding-3-small"
)

Settings.embed_model = embed_model

print("Embedding model configured.")
print("Model:", embed_model.model_name)


In [ ]:
import os

# Create a local "documents" folder with a few small sample text files.
# In a real project you would point SimpleDirectoryReader at your own
# folder of PDFs, .txt, .docx, etc. instead of generating these.
os.makedirs("documents", exist_ok=True)

documents_data = {
    "python.txt": """
Python is a high-level programming language.
Python is widely used in data science, machine learning,
web development and automation.

Python supports multiple programming paradigms including
object-oriented, procedural and functional programming.
""",

    "machine_learning.txt": """
Machine Learning is a branch of Artificial Intelligence.

Machine learning algorithms learn patterns from data
and use those patterns to make predictions or decisions.

There are three common types of machine learning:
supervised learning, unsupervised learning and reinforcement learning.
""",

    "rag.txt": """
Retrieval Augmented Generation, commonly called RAG,
is a technique used to improve the responses of Large Language Models.

A RAG system retrieves relevant information from an external
knowledge source and provides that information to the LLM.
The LLM then generates an answer using the retrieved context.
"""
}

for filename, content in documents_data.items():
    with open(f"documents/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

print("Documents created successfully!")
print(os.listdir("documents"))


In [ ]:
# Data Ingestion
#
# SimpleDirectoryReader walks the given folder and loads every supported
# file (.txt, .pdf, .docx, etc.) into a list of Document objects. Each
# Document holds the raw text plus metadata like the source filename.
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_dir="documents"
).load_data()

print("Number of documents:", len(documents))


In [ ]:
# Inspect one loaded document to confirm the text and metadata came
# through correctly before we move on to chunking.
print("Source file:", documents[1].metadata.get("file_name"))
print("-" * 60)
print(documents[1].text)


In [ ]:
# Chunking (Node Parsing)
#
# LLMs and embedding models work best on short, focused pieces of text
# rather than entire documents. SentenceSplitter breaks each Document
# into smaller "nodes" along sentence boundaries.
#
# - chunk_size=100    -> roughly the max size (in tokens) per chunk
# - chunk_overlap=20  -> chunks share some overlapping text so a fact
#                        that spans a chunk boundary isn't lost entirely
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(
    chunk_size=100,
    chunk_overlap=20
)

nodes = splitter.get_nodes_from_documents(documents)

print("Number of nodes:", len(nodes))


In [ ]:
# Preview the first few chunks to see how the documents were split.
for i, node in enumerate(nodes[:5]):
    print("=" * 60)
    print("NODE:", i)
    print(node.text)


In [ ]:
# Generate Embeddings (demo on a single chunk)
#
# Node
#  ↓
# Embedding Model
#  ↓
# Vector
#
# This just demonstrates what an embedding looks like for one chunk
# before we embed *all* chunks as part of building the index below.
sample_text = nodes[1].text

embedding = embed_model.get_text_embedding(sample_text)

print("Text:")
print(sample_text)

print("\nEmbedding (first 10 dimensions):")
print(embedding[:10])

print("\nEmbedding dimension:")
print(len(embedding))


## Step: Building the Vector Store Index

Now that we have chunked nodes, we embed *all* of them and store the
resulting vectors in a searchable index. `VectorStoreIndex` automatically
uses the embedding model we set on `Settings.embed_model` earlier, so we
don't need to pass it explicitly.

Behind the scenes this:
1. Sends each node's text to the embedding model
2. Stores the resulting vector alongside the node's text and metadata
3. Builds a data structure that supports fast "find the most similar
   vectors to this query" lookups

In [ ]:
# Build the vector index from our chunked nodes.
# This is the step the original notebook stopped just before —
# everything from here on turns the embeddings into an actual
# question-answering system.
index = VectorStoreIndex(nodes)

print("Vector index built successfully!")
print("Number of indexed nodes:", len(nodes))


## Step: Persisting the Index (optional but recommended)

Building the index re-embeds every chunk, which costs time and API
calls. Persisting it to disk lets us reload it instantly next time
without recomputing embeddings.

In [ ]:
# Save the index to a local "storage" folder.
PERSIST_DIR = "storage"
index.storage_context.persist(persist_dir=PERSIST_DIR)
print(f"Index persisted to '{PERSIST_DIR}/'.")

# --------------------------------------------------------------
# To reload this index later WITHOUT re-embedding your documents,
# use the following instead of rebuilding it from scratch:
#
# from llama_index.core import StorageContext, load_index_from_storage
# storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
# index = load_index_from_storage(storage_context)
# --------------------------------------------------------------


## Step: Creating the Query Engine

A query engine wraps the index with the retrieval + generation logic:

```
User question
     ↓
Embed the question
     ↓
Find the top-k most similar chunks in the index   (Retrieval)
     ↓
Pass question + retrieved chunks to the LLM        (Augmented Generation)
     ↓
Return the final answer
```

`similarity_top_k=3` means we retrieve the 3 most relevant chunks for
every question before asking the LLM to answer.

In [ ]:
# Turn the index into a query engine — this is the object we actually
# "chat" with.
query_engine = index.as_query_engine(similarity_top_k=3)

print("Query engine ready!")


## Step: Asking a Single Question

Let's test the full RAG pipeline on one question and inspect not just
the answer, but *which* chunks were retrieved to produce it — this is
what makes RAG transparent and debuggable compared to asking an LLM
"cold" with no context.

In [ ]:
# Ask a question that should be answerable from our sample documents.
question = "What is machine learning?"
response = query_engine.query(question)

print("Q:", question)
print("-" * 60)
print("A:", response)

print("\n" + "=" * 60)
print("Retrieved context used to generate this answer:")
print("=" * 60)
for i, source_node in enumerate(response.source_nodes):
    print(f"\n[Chunk {i}] (similarity score: {source_node.score:.4f})")
    print(f"From file: {source_node.metadata.get('file_name')}")
    print(source_node.text)


## Step: Testing Multiple Questions

Let's run a small batch of questions covering all three sample
documents to confirm retrieval is pulling from the right files.

In [ ]:
# A small test set covering each of our three sample documents.
test_questions = [
    "What is Python used for?",
    "What are the three types of machine learning?",
    "What is Retrieval Augmented Generation (RAG)?",
]

for question in test_questions:
    response = query_engine.query(question)
    print("=" * 60)
    print("Q:", question)
    print("-" * 60)
    print("A:", response)
    print()


## Step: Simple Reusable Chat Function

Finally, let's wrap everything in a small `ask()` helper so the RAG
system can be reused with a single function call, plus an optional
`chat()` loop for interactive use directly in the notebook.

In [ ]:
def ask(question: str, show_sources: bool = False) -> str:
    """
    Ask a question of the RAG system and return the answer.

    Args:
        question: the natural-language question to ask.
        show_sources: if True, also prints the retrieved chunks
                       that were used to generate the answer.

    Returns:
        The answer text as a string.
    """
    response = query_engine.query(question)

    if show_sources:
        print("Retrieved context:")
        for i, source_node in enumerate(response.source_nodes):
            print(f"  [{i}] {source_node.metadata.get('file_name')} "
                  f"(score: {source_node.score:.4f})")

    return str(response)


def chat():
    """
    Simple interactive chat loop for this notebook.

    Type a question and press Enter to get an answer.
    Type 'exit' or 'quit' to stop.

    Note: this uses input(), so it only works when run interactively
    in a live notebook session (not during automated/batch execution).
    """
    print("🤖 RAG Chatbot ready! Type 'exit' to stop.\n")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in ("exit", "quit"):
            print("Goodbye!")
            break
        if not user_input:
            continue
        answer = ask(user_input)
        print("Bot:", answer, "\n")


# Example of using ask() directly (safe to run non-interactively):
print(ask("What is LlamaIndex used for according to these documents?"))

# To chat interactively, uncomment the line below and run this cell:
# chat()
